<a href="https://colab.research.google.com/github/mrshibly/MiniGPT-from-Scratch/blob/main/notebooks/Colab_Training_Template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniGPT: 50M Parameter Training Run

This notebook will download, install, and train the MiniGPT model from scratch using a GPU.
**Make sure to set your runtime to GPU (T4 is fine).**

In [1]:
# 1. Clone the repository
# Check if folder exists, if not clone it
import os
if not os.path.exists('MiniGPT-from-Scratch'):
    !git clone https://github.com/mrshibly/MiniGPT-from-Scratch.git

%cd MiniGPT-from-Scratch
!pip install -r requirements.txt

Cloning into 'MiniGPT-from-Scratch'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 76 (delta 23), reused 64 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 3.31 MiB | 8.74 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/MiniGPT-from-Scratch


## Step 1: Download & Preprocess Data
We will download ~500MB of the FineWeb-Edu dataset.

In [2]:
# The code on GitHub is already set to 500MB, but let's be sure
!sed -i 's/max_bytes = 10 \* 1024 \* 1024/max_bytes = 500 \* 1024 \* 1024/' src/datasets/download_fineweb.py

# Run Data Pipeline
!python src/datasets/download_fineweb.py
!python src/datasets/clean_text.py
!python src/tokenizer/train_tokenizer.py
!python src/datasets/prepare_data.py

README.md: 26.4kB [00:00, 53.5MB/s]
Resolving data files: 100% 2410/2410 [00:00<00:00, 9709.15it/s]
Saving ~500MB of text to data/raw/sample.txt...
Downloading: 524MB [00:44, 11.8MB/s]               
Successfully saved 500.00 MB of text!
terminate called without an active exception
Reading from data/raw/sample.txt...
Writing to data/processed/clean_sample.txt...
Cleaning: 100% 525M/525M [00:40<00:00, 13.0MB/s]

Done! Kept: 109130 docs. Removed: 0 docs.
Clean file size: 500.20 MB
Training BPE tokenizer on data/processed/clean_sample.txt...
[00:00:01] Tokenize words                 ██████████████████ 908726   /   908726
[00:00:03] Count pairs                    ██████████████████ 908726   /   908726
[00:00:06] Compute merges                 ██████████████████ 16127    /    16127
Successfully trained and saved tokenizer to data/tokenizer/tokenizer.json
Vocabulary size: 16384
Loading tokenizer from data/tokenizer/tokenizer.json...
Reading data from data/processed/clean_sample.txt...
Loaded

## Step 2: Configure & Train

Switching to the 50M parameter config.

In [3]:
# Configure standard run
!sed -i 's/config = MiniGPTConfig.tiny()/config = MiniGPTConfig.standard()/' src/train/train.py
!sed -i 's/max_steps = 2000/max_steps = 10000/' src/train/train.py
!sed -i 's/batch_size = 16/batch_size = 8/' src/train/train.py

# Launch training
!python src/train/train.py

Targeting device: cuda
Loading data/processed/train.bin via memmap...
Dataset contains 107,218,358 tokens.
Loading data/processed/val.bin via memmap...
Dataset contains 12,111,648 tokens.
Total Parameters: 27.54 M | Using Precision: bfloat16

Starting Training...

Step    0 | Train Loss: 9.8207 | Val Loss: 9.8204
Step    0 | Loss: 9.8320 | LR: 2.00e-06 | Time: 6016.98ms | Tok/sec: 680.74
Step   20 | Loss: 8.9973 | LR: 4.20e-05 | Time: 8340.57ms | Tok/sec: 491.09
Step   40 | Loss: 8.5539 | LR: 8.20e-05 | Time: 8476.38ms | Tok/sec: 483.23
Step   60 | Loss: 7.9999 | LR: 1.22e-04 | Time: 8524.50ms | Tok/sec: 480.50
Step   80 | Loss: 7.5818 | LR: 1.62e-04 | Time: 8591.16ms | Tok/sec: 476.77
Step  100 | Loss: 7.4824 | LR: 2.02e-04 | Time: 8781.75ms | Tok/sec: 466.42
Step  120 | Loss: 7.6266 | LR: 2.42e-04 | Time: 8908.57ms | Tok/sec: 459.78
Step  140 | Loss: 7.2752 | LR: 2.82e-04 | Time: 8980.44ms | Tok/sec: 456.10
Step  160 | Loss: 7.1635 | LR: 3.22e-04 | Time: 9182.64ms | Tok/sec: 446.06
S

## Step 3: Evaluate


In [4]:
!python src/model/evaluate.py

Loading checkpoint from checkpoints/ckpt.pt to cuda...
Loading data/processed/val.bin via memmap...
Dataset contains 12,111,648 tokens.
Evaluating Perplexity over 50 batches...

--- EVALUATION RESULTS ---
Validation Cross-Entropy Loss: 4.8415
Validation Perplexity:       126.6581
--------------------------

Good! The model has definitely learned basic English grammar and vocabulary.
